# 02 - Silver: Hospital Dimension

**Pipeline:** Bronze -> Silver  
**Source:** AIHW MyHospitals `/reporting-units` API (WA hospitals)  
**Target:** `silver.dim_hospitals` (Delta table)  

Steps:
1. Read `wa_hospitals.json` from bronze layer (absolute OneLake path)
2. Flatten `reporting_unit_summary` struct
3. Extract health service (LHN) from nested `mapped_reporting_units`
4. Profile and write as Delta dimension table

In [ ]:
# ------------------------------------------------------------------
# Config - absolute OneLake paths
# ------------------------------------------------------------------
WORKSPACE_ID = "e53f915a-de32-40a9-9b16-af4486796bbe"
LAKEHOUSE_ID = "6383e12e-91b9-4df2-99c5-06c9597bc27e"
ONELAKE_ROOT = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"

from pyspark.sql.functions import col, explode

print(f"OneLake root: {ONELAKE_ROOT}")

In [ ]:
# ------------------------------------------------------------------
# 1. Read WA hospital reference data from bronze layer
#    Ingested by: scripts/ingest_bronze.py (AIHW /reporting-units)
# ------------------------------------------------------------------
path = f"{ONELAKE_ROOT}/Files/bronze/aihw/reporting_units/wa_hospitals.json"

wa_raw = spark.read.option("multiline", "true").json(path)

wa = wa_raw.select(explode(col("result")).alias("h")).select(
    col("h.reporting_unit_code").alias("hospital_code"),
    col("h.reporting_unit_name").alias("hospital_name"),
    col("h.latitude").cast("double"),
    col("h.longitude").cast("double"),
    col("h.private"),
    col("h.closed"),
    col("h.mapped_reporting_units").alias("mappings")
)

print(f"Total WA hospitals loaded: {wa.count()}")
wa.printSchema()

In [ ]:
# ------------------------------------------------------------------
# 2. Extract health service (LHN) from nested mapped_reporting_units
#    mapped_reporting_unit_code == 'H_LHN' identifies the LHN mapping
# ------------------------------------------------------------------
lhn_mapping = (
    wa.select("hospital_code", explode("mappings").alias("m"))
    .filter(col("m.map_type.mapped_reporting_unit_code") == "H_LHN")
    .select(
        "hospital_code",
        col("m.mapped_reporting_unit.reporting_unit_name").alias("health_service")
    )
)

print(f"Hospitals with LHN mapping: {lhn_mapping.count()}")
lhn_mapping.show(5)

# Join LHN back to hospital base
dim_hospitals = (
    wa.drop("mappings")
    .join(lhn_mapping, "hospital_code", "left")
)

print(f"\nDim hospitals records: {dim_hospitals.count()}")
dim_hospitals.show(10, truncate=False)

In [ ]:
# ------------------------------------------------------------------
# 3. Profile - health service distribution and coordinate coverage
# ------------------------------------------------------------------
print("Health services (Local Hospital Networks):")
dim_hospitals.groupBy("health_service").count().orderBy("count", ascending=False).show()

total       = dim_hospitals.count()
has_coords  = dim_hospitals.filter(col("latitude").isNotNull()).count()
is_private  = dim_hospitals.filter(col("private") == True).count()
is_closed   = dim_hospitals.filter(col("closed") == True).count()

print(f"Total:            {total}")
print(f"With coordinates: {has_coords} ({has_coords/total*100:.0f}%)")
print(f"Private:          {is_private}")
print(f"Closed:           {is_closed}")

In [ ]:
# ------------------------------------------------------------------
# 4. Write silver Delta table
# ------------------------------------------------------------------
dim_hospitals.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_hospitals")

print("silver.dim_hospitals written successfully")
print(f"Total hospitals: {spark.table('silver.dim_hospitals').count()}")
spark.table("silver.dim_hospitals").show(5)